In [ ]:
import pandas as pd
import numpy as np
import pyro
import torch

np.random.seed(13625442)

# 1: Load the Dataset

In [2]:
ratings_df = pd.read_csv("Dataset/jester_ratings.csv")  
jokes_df = pd.read_csv("Dataset/jester_items.csv")

print(f"Ratings: {ratings_df.shape}\nJokes: {jokes_df.shape}")
print(ratings_df.head())
print(jokes_df.head())

Ratings: (1761439, 3)
Jokes: (150, 2)
   userId  jokeId  rating
0       1       5   0.219
1       1       7  -9.281
2       1       8  -9.281
3       1      13  -6.781
4       1      15   0.875
   jokeId                                           jokeText
0       1  A man visits the doctor. The doctor says "I ha...
1       2  This couple had an excellent relationship goin...
2       3  Q. What's 200 feet long and has 4 teeth? \n\nA...
3       4  Q. What's the difference between a man and a t...
4       5  Q.\tWhat's O. J. Simpson's Internet address? \...


# 2: Filter the Dataset to be smaller 

In [3]:
JOKE_COUNT = 50
USER_COUNT = 1000

#order jokes by the jokes that are rated the most often and take the top JOKE_COUNT jokes
top_jokeIds = ratings_df['jokeId'].value_counts().head(JOKE_COUNT).index.tolist()

#copy the dataset with only the jokes from last step
filtered_df = ratings_df[ratings_df['jokeId'].isin(top_jokeIds)].copy()

#order the users by how many of the jokes they rated in our filtered_df, 
#then keep the users that rated more than 4/5ths of the jokes
user_counts = filtered_df['userId'].value_counts()
active_users = user_counts[user_counts >= (JOKE_COUNT//5)*4].index
filtered_df = filtered_df[filtered_df['userId'].isin(active_users)]

#rearrange the df such that every row is a single user and every column is a joke
pivot_df = filtered_df.pivot(index='userId', columns='jokeId', values='rating')[:USER_COUNT]

#mask out NANs 
mask_np = pivot_df.notna().astype(float).values
mask_tensor = torch.tensor(mask_np, dtype=torch.float32)

def binarize_rating(val):
    """Convert positive values to 1.0 and negative values to 0.0."""
    return 1.0 if val > 0.0 else 0.0

#create tensor for model
binary_ratings_np = pivot_df.map(binarize_rating).values
ratings_tensor = torch.tensor(binary_ratings_np, dtype=torch.float32)

#reconstruct jokeText for pretty printing later
ordered_jokeIds = pivot_df.columns.tolist()
top_jokes_text = [
    jokes_df[jokes_df['jokeId'] == j_id]['jokeText'].values[0] 
    for j_id in ordered_jokeIds
]

print(f"Data ready! Matrix shape: {ratings_tensor.shape} (Users x Jokes)")

Data ready! Matrix shape: torch.Size([1000, 50]) (Users x Jokes)


# 2.5 Print the Funniest Joke (for fun)

In [4]:

joke_scores = binary_ratings_np.sum(axis=0)
sorted_joke_indices = np.argsort(joke_scores)

funniest_joke_idx = sorted_joke_indices[-1]
highest_score = joke_scores[funniest_joke_idx]

print(f"Funniest Joke:\n{top_jokes_text[funniest_joke_idx]}")

Funniest Joke:
A radio conversation of a US naval 
ship with Canadian authorities ... 

Americans: Please divert your course 15 degrees to the North to avoid a
collision.

Canadians: Recommend you divert YOUR course 15 degrees to the South to 
avoid a collision.

Americans: This is the Captain of a US Navy ship.  I say again, divert 
YOUR course.

Canadians: No.  I say again, you divert YOUR course.

Americans: This is the aircraft carrier USS LINCOLN, the second largest ship in the United States' Atlantic Fleet. We are accompanied by three destroyers, three cruisers and numerous support vessels. I demand that you change your course 15 degrees north, that's ONE FIVE DEGREES NORTH, or counter-measures will be undertaken to ensure the safety of this ship.

Canadians: This is a lighthouse.  Your call.



# 3: Define the model

In [10]:
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.infer.autoguide import AutoNormal
from pyro.optim import Adam
import torch

def ising_jester_model(user_ratings, mask):
    n_users, n_jokes = user_ratings.shape
    
    #instantiate model parameters
    h_prior = dist.Normal(torch.zeros(n_jokes), torch.ones(n_jokes))
    J_prior = dist.Normal(torch.zeros(n_jokes, n_jokes), torch.ones(n_jokes, n_jokes) * 0.1)
    
    h = pyro.sample("h", h_prior.to_event(1))
    J_raw = pyro.sample("J_raw", J_prior.to_event(2))
    
    #create symmetrical matrix and eliminate diagonal entries
    J_sym = (J_raw + J_raw.T) / 2.0
    J = J_sym - torch.diag(torch.diag(J_sym)) 
    
    with pyro.plate("users", n_users):
        observed_states = user_ratings * mask
        
        #convert {0,1} -> {-1,1}
        spins = (2.0 * observed_states - 1.0) * mask
        
        #effective field for ising learning
        effective_field = 2.0 * h + 2.0 * torch.matmul(spins, J)
        prob_funny = torch.special.expit(effective_field)

        masked_dist = dist.Bernoulli(probs=prob_funny).mask(mask.bool()).to_event(1)
        pyro.sample("obs_ratings", masked_dist, obs=user_ratings)



Training the pure-tensor Ising model...
Step 0000 | Negative ELBO Loss: 38173.6633
Step 0200 | Negative ELBO Loss: 21583.9384
Step 0400 | Negative ELBO Loss: 21198.0110
Step 0600 | Negative ELBO Loss: 20813.0666
Step 0800 | Negative ELBO Loss: 20706.8629
Step 1000 | Negative ELBO Loss: 20817.2204
Step 1200 | Negative ELBO Loss: 20794.0141
Step 1400 | Negative ELBO Loss: 20886.7242
Step 1600 | Negative ELBO Loss: 20855.5285
Step 1800 | Negative ELBO Loss: 20707.5670
Step 2000 | Negative ELBO Loss: 20916.6835
Step 2200 | Negative ELBO Loss: 20963.0738
Step 2400 | Negative ELBO Loss: 20825.8615
Step 2600 | Negative ELBO Loss: 20765.4668
Step 2800 | Negative ELBO Loss: 21095.1564
Step 3000 | Negative ELBO Loss: 20648.4425
Step 3200 | Negative ELBO Loss: 20709.7344
Step 3400 | Negative ELBO Loss: 20827.1325
Step 3600 | Negative ELBO Loss: 20811.6162
Step 3800 | Negative ELBO Loss: 21063.8520
Step 4000 | Negative ELBO Loss: 20941.6602
Step 4200 | Negative ELBO Loss: 20961.6176
Step 4400 | Ne

In [24]:
total_users, total_jokes = ratings_tensor.shape
base_train_mask = mask_tensor.clone()
test_targets = []
STEPS = 1000

#create a test set of 1 joke per user
for u in range(total_users):
    rated_indices = torch.where(mask_tensor[u] == 1.0)[0]
    target_joke = rated_indices[np.random.randint(0, len(rated_indices))].item()
    base_train_mask[u, target_joke] = 0.0
    test_targets.append((u, target_joke, ratings_tensor[u, target_joke].item()))

keep_rates = [0.10, 0.25, 0.5, 0.75, 1.0]
results_log = []


for keep_rate in keep_rates:
    #generate a mask, keeping only keep_rate% of entries
    random_dropout = (torch.rand(total_users, total_jokes) < keep_rate).float()
    sparse_train_mask = base_train_mask * random_dropout
    
    #get a count of the ratings
    ratings_count = int(sparse_train_mask.sum().item())
    
    #reset the model
    pyro.clear_param_store()
    guide = AutoNormal(ising_jester_model)
    optimizer = Adam({"lr": 0.01})
    svi = SVI(ising_jester_model, guide, optimizer, loss=Trace_ELBO())
    
    #train
    for step in range(STEPS):  
        loss = svi.step(ratings_tensor, sparse_train_mask)
        
    #extract the parameters
    post_means = guide.median()
    learned_h = post_means["h"]
    learned_J_raw = post_means["J_raw"]
    learned_J = (learned_J_raw + learned_J_raw.T) / 2.0 - torch.diag(torch.diag((learned_J_raw + learned_J_raw.T) / 2.0))
    
    #evaluate
    obs_states = ratings_tensor * sparse_train_mask
    spins = (2.0 * obs_states - 1.0) * sparse_train_mask
    eff_field = 2.0 * learned_h + 2.0 * torch.matmul(spins, learned_J)
    probs = torch.special.expit(eff_field)
    
    #accuracy
    correct, total = 0, 0
    for u, j, actual in test_targets:
        pred = 1.0 if probs[u, j].item() >= 0.5 else 0.0
        if pred == actual:
            correct += 1
        total += 1
            
    accuracy = (correct / total * 100) if total > 0 else 0
    results_log.append({"Keep Rate": keep_rate, "Active Ratings": ratings_count, "Accuracy": accuracy})
    
    print(f"Kept {keep_rate*100}% Data | Ratings Used: {ratings_count:>5} | Test Accuracy: {accuracy:.2f}%")


Kept 10.0% Data | Ratings Used:  4648 | Test Accuracy: 75.50%
Kept 25.0% Data | Ratings Used: 11425 | Test Accuracy: 75.90%
Kept 50.0% Data | Ratings Used: 23061 | Test Accuracy: 78.90%
Kept 75.0% Data | Ratings Used: 34593 | Test Accuracy: 79.40%
Kept 100.0% Data | Ratings Used: 46185 | Test Accuracy: 80.50%


In [26]:
funniest_joke_idx = torch.argmax(learned_h).item()
funniest_joke_bias = learned_h[funniest_joke_idx].item()

print(f"Funniest Joke(model edition)\n{top_jokes_text[funniest_joke_idx]}")
print(f"Intrinsic Bias (h): {funniest_joke_bias:.4f}")



Funniest Joke(model edition)
A couple of hunters are out in the woods in the deep south when one of them falls to the ground. He doesn't seem to be breathing, and his eyes are rolled back in his head. The other guy whips out his cell phone and calls 911. He gasps to the operator, "My friend is dead! What can I do?" The operator, in a calm and soothing voice, says, "Alright, take it easy. I can help. First, let's make sure he's dead." There is silence, and then a gun shot is heard. The hunter comes back on the line. "Okay. Now what??"
Intrinsic Bias (h): 0.3963
